<div style="background: linear-gradient(120deg,#0f172a,#1e3a5f); border-radius:16px; padding:36px 40px; color:#e2e8f0;">
<h1 style="margin:0; font-size:2.1em; color:#ffffff;">⚡ RAG の基礎</h1>
<h3 style="margin:6px 0 0 0; font-weight:400; color:#93c5fd;">セッション 1 / 4 — なぜ「検索」が必要なのか、そしてどこで壊れるのか</h3>
<p style="margin-top:18px; color:#cbd5e1;">NordWind Energy ワークショップシリーズ · Retrieval, Vectors & Graphs</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noctetemp/nordwind-workshop/blob/main/session1_rag_foundations_ja.ipynb)

## 🗺️ 今日の道のり

今日は「はしご」を一段ずつ登ります。各ステップが**次のステップの必要性**を生み出す構成です。終わる頃には、RAG は誰かに渡されたレシピではなく、**必然**として腑に落ちているはずです。

* **🔤 トークン** — LLM が実際に読んでいる単位。ライブデモ: 同じ意味の文を英語と日本語でトークン化し、数を比較します(ネタバレ: 日本語には「トークン税」がかかります)。
* **🧠 コンテキストウィンドウ** — モデルの「作業記憶」。NordWind の全コーパスをトークンで測り、Claude のウィンドウと比較します。そして実在企業の Wiki にスケールすると、ウィンドウを **200倍** あふれることを目撃します。
* **📉 「ウィンドウを大きくすればいい」が通用しない理由** — Attention のコスト、そして *lost in the middle* 問題。長いコンテキストの中に事実を隠し、**どこに隠すか**で回答の質が揺れる様子を観察します。
* **🎓 なぜファインチューニングではないのか** — コスト、鮮度、出典の欠如。デモ不要の正直な議論を5分。
* **🔍 だからこそ検索(Retrieval)** — 約40行でナイーブな RAG パイプラインをゼロから構築します。チャンク → エンベディング → 検索 → 生成。そして、ちゃんと*動きます*。
* **💥 破綻** — その自慢の RAG に、ある特定の質問を投げます……そして盛大に転びます。*なぜ*転ぶのかを理解することが、セッション 2・3・4 への扉です。

> **4セッションを通して使う世界:** *NordWind Energy* — 8チーム、30名のエンジニア、15のサービス、20のインシデント、65の社内ドキュメントを持つ架空の電力会社です。今日作るものはすべて再利用され、アップグレードされ、最終的にはナレッジグラフになります。

## 0 · セットアップ

依存パッケージ2つとデータセット1つだけです。実行して、インストールの間(Colab で約1分)ひと息どうぞ。

In [ ]:
%pip -q install anthropic sentence-transformers tiktoken matplotlib
print("✅ Dependencies installed")

In [ ]:
# --- Download the NordWind dataset -------------------------------------
import json, urllib.request, pathlib

DATA_URL = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/documents.jsonl"
LOCAL = pathlib.Path("documents.jsonl")

if not LOCAL.exists():
    try:
        urllib.request.urlretrieve(DATA_URL, LOCAL)
    except Exception:
        print("⚠️ Could not download — upload documents.jsonl manually via the Files panel ⬅️")

docs = [json.loads(line) for line in LOCAL.read_text().splitlines()]
print(f"📚 Loaded {len(docs)} NordWind documents")
from collections import Counter
print(Counter(d['doc_type'] for d in docs))

In [ ]:
# --- Anthropic API key ---------------------------------------------------
# In Colab: click the 🔑 icon in the left sidebar → add secret ANTHROPIC_API_KEY
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("🔑 API key loaded from Colab secrets")
except Exception:
    print("⚠️ No Colab secret found — LLM cells will be skipped gracefully")

import anthropic
MODEL = "claude-sonnet-4-6"   # switch to a smaller model to reduce cost
client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None

def ask_claude(prompt, system="You are a helpful assistant.", max_tokens=600):
    if client is None:
        return "[LLM skipped — no API key configured]"
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                               system=system,
                               messages=[{"role": "user", "content": prompt}])
    return r.content[0].text

コーパスを実感するために、測定を始める前にドキュメントを1つ覗いてみましょう:

In [ ]:
sample = next(d for d in docs if d['doc_id'] == 'PM-INC-2102')
print(sample['text'][:900])

---
## 1 · 🔤 トークン — モデルが実際に読んでいるもの

LLM は文字や単語を読んでいるわけではありません。読んでいるのは**トークン** — 学習データから獲得した固定語彙のかたまりです。コスト、速度、メモリに関するすべてはトークン単位で数えられます。

頻出する英単語はたいてい 1 トークンです。珍しい単語は分割されます。そして、トークナイザの学習データに占める割合が少ない言語は**トークン税**を払うことになります — 実際に見てみましょう。

In [ ]:
import tiktoken
# We use a well-known open tokenizer to demonstrate the concept.
# Every model family (Claude, GPT, Llama...) has its own tokenizer, but the behaviour is universal.
enc = tiktoken.get_encoding("cl100k_base")

pairs = [
    ("English",  "The payment gateway timed out during the evening peak."),
    ("Japanese", "決済ゲートウェイは夜間のピーク時にタイムアウトしました。"),
]
for lang, sentence in pairs:
    toks = enc.encode(sentence)
    print(f"{lang:9s} | {len(sentence):3d} chars → {len(toks):3d} tokens")
    pieces = [enc.decode_single_token_bytes(t) for t in toks[:14]]
    print("          |", [p.decode("utf-8", errors="replace") for p in pieces], "…\n")

# One kanji under the microscope: BPE tokenizers work on BYTES, not characters
kanji = "決"
print(f"'{kanji}' = 1 character = {len(kanji.encode('utf-8'))} bytes in UTF-8 = {len(enc.encode(kanji))} tokens")
print("raw bytes of each token:", [enc.decode_single_token_bytes(t) for t in enc.encode(kanji)])

**この � 記号はバグではありません — これこそが教材です。** トークナイザは UTF-8 の*バイト列*に対して動作します。「決」のような漢字は 3 バイトを占め、トークナイザはしばしばこれを*文字の途中で* 2 つのトークンに分割します。片方のトークンだけをデコードすると不正な UTF-8 になり、ターミナルはそれを � として表示するのです。英単語はトークンにきれいに対応する一方、日本語の文字はバイト断片に切り刻まれます。*これ*がトークン税の力学的な正体です。

**会場への問いかけ:** 日本語の文は同じ内容を伝えているのに、トークン数は約2倍です。日英バイリンガルの組織にとって、これはコンテキスト予算・レイテンシ・コストに直結します — 教養クイズではなく実務の話です。

さて、このワークショップ全体を駆動する問いに進みます: *モデルは一度にどれだけ読めるのか?*

---
## 2 · 🧠 コンテキストウィンドウ — 「作業記憶」であって「知識」ではない

**コンテキストウィンドウ**とは、1リクエストでモデルが注意を向けられるトークンの上限です — プロンプトと応答の*合計*です。長期知識(図書館)ではなく、作業記憶(机の上)だと考えてください。

Claude のウィンドウは約 **20万トークン**。巨大に聞こえますね。では、私たちの小さな架空企業を測って比べてみましょう。

In [ ]:
corpus_tokens = sum(len(enc.encode(d['text'])) for d in docs)
print(f"NordWind corpus: {len(docs)} docs, {corpus_tokens:,} tokens")

CLAUDE_WINDOW = 200_000
REAL_CONFLUENCE = 40_000_000   # a realistic internal wiki for a 150-engineer org

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.2))
bars = [("NordWind corpus\n(65 docs)", corpus_tokens, "#34d399"),
        ("Claude context window", CLAUDE_WINDOW, "#60a5fa"),
        ("A real company wiki", REAL_CONFLUENCE, "#f87171")]
ax.barh([b[0] for b in bars], [b[1] for b in bars], color=[b[2] for b in bars])
ax.set_xscale("log"); ax.set_xlabel("tokens (log scale!)")
for i, (_, v, _) in enumerate(bars):
    ax.text(v, i, f"  {v:,}", va="center")
ax.set_title("Working memory vs. what you actually know")
plt.tight_layout(); plt.show()

チャートを注意深く読んでください — 軸は**対数**です。私たちのおもちゃのコーパスはウィンドウに収まります(考える余白がわずかに残る程度に)。しかし*実在する*ナレッジベースは**ウィンドウの200倍**。収まりません。今後も収まることはありません。

(そして正直に言うと — この*ワークショップ用*コーパスが小さいのは意図的です。30人が無料の Colab で数秒のうちにエンベディングを終えられるサイズにしてあります。緑のバーなら今日でも1つのプロンプトに詰め込めてしまいます。この反論にはセッションの最後で正面から向き合います。今日構築するあらゆる技術は、緑ではなく**赤のバー**を基準に評価してください。)

つまり、素朴な計画 — *「全部プロンプトに貼り付ければいい」* — は、現実の組織においては最初から詰んでいます。でも待ってください、ウィンドウは拡大し続けていますよね……4000万トークンのウィンドウを待てばいいのでは?

---
## 3 · 📉 「大きくすればいい」では救われない理由

厄介な問題が2つあります:

1. **コストとレイテンシはコンテキストに比例して増えます。** N トークンに対する Attention のコストは最悪 O(N²) — しかも入力トークンには*リクエストのたびに*課金されます。「請求書の支払期限はいつ?」に答えるために 4000万トークンを送るのは、一文の引用を求めた人に図書館ごと郵送するようなものです。
2. **Lost in the middle。** 内容がウィンドウに収まる場合でも、モデルはコンテキストの**先頭**と**末尾**付近の情報を、**中間**に埋もれた情報よりよく思い出します。検索品質は、あなたの見えない場所で静かに劣化します。

2つ目を自分たちのコーパスでライブ検証しましょう: NordWind ドキュメントの大きな塊の中に特定の事実を1つ隠します — 先頭、中間、末尾に — そして Claude に探させます。

In [ ]:
# --- Needle in a haystack ------------------------------------------------
NEEDLE = ("INTERNAL MEMO: The NordWind disaster-recovery passphrase rotation "
          "is scheduled for the 14th of every month at 03:00 UTC.")
QUESTION = "When is the disaster-recovery passphrase rotation scheduled?"

haystack = "\n\n".join(d['text'] for d in docs if d['doc_type'] in ('runbook','overview','adr'))
print(f"Haystack size: {len(enc.encode(haystack)):,} tokens")

def hide_needle(position):
    if position == "start":  return NEEDLE + "\n\n" + haystack
    if position == "end":    return haystack + "\n\n" + NEEDLE
    half = len(haystack) // 2
    return haystack[:half] + "\n\n" + NEEDLE + "\n\n" + haystack[half:]

for pos in ["start", "middle", "end"]:
    context = hide_needle(pos)
    answer = ask_claude(
        f"Answer strictly from the documents below.\n\n<documents>\n{context}\n</documents>\n\nQuestion: {QUESTION}",
        max_tokens=100)
    print(f"[needle at {pos:6s}] → {answer.strip()[:140]}")

> 🎤 **ファシリテーター向けメモ:** 最近のモデルはこの小規模なテストに*合格*することが多いです — それで問題ありませんし、正直にそう伝えてください。この現象はコンテキスト長とノイズ密度が増すほど顕著になることが文献で示されています。このデモの役目は「**位置とノイズが効いてくる**」という直観を植え付けること、そして「詰め込み」は品質を監視できない戦略だと理解してもらうことです。本番でぐらついたら? むしろ好都合です。

まとめ: 全部は収まらない。収まったとしても想起は一様ではない。では、知識を重みに焼き込むのはどうでしょう?

---
## 4 · 🎓 なぜファインチューニングではないのか

ファインチューニングはモデルの**重み**を変更します。*振る舞い*を変えるには正しい道具です — 口調、フォーマット、ドメイン特有の文体。しかし*事実*を注入するには間違った道具です:

| | ファインチューニング | 検索 (Retrieval) |
|---|---|---|
| 新しいドキュメントが届いた | 再学習(数時間、高コスト) | インデックスに追加(数秒) |
| 「その答えの出典は?」 | 🤷 出典なし | 該当チャンクを引用できる |
| 事実の削除・訂正 | ほぼ不可能 | チャンクを削除・更新するだけ |
| ユーザーごとのアクセス制御 | 不可 | クエリ時にフィルタ |
| 知識がないときの挙動 | 自信満々に間違える | 「見つからない」と言える |

**知識**はモデルの*外*に、自分たちが管理するストアに置きたい — そして質問された瞬間に取ってくる。

その「取ってくる」という一語がすべてです。ようこそ、**Retrieval-Augmented Generation** へ。

---
## 5 · 🔍 ナイーブ RAG をゼロから作る

パイプラインの全体像です。これを残り3セッションかけて磨いていきます:

```
 ingest → chunk → embed → index → RETRIEVE → augment prompt → generate
```

今日は最もシンプルで、しかし誠実なバージョンを作ります:
- **チャンク分割**: 各ドキュメントをオーバーラップ付きのウィンドウに分割
- **エンベディング**: ローカルモデルで各チャンクをベクトル化(API 不要)
- **インデックス**: ただの numpy 行列です 😄(なぜこれが破綻するかはセッション2で)
- **検索**: コサイン類似度で top-k
- **生成**: Claude が*取得したチャンクのみ*を使って回答

In [ ]:
# --- Chunking ------------------------------------------------------------
def chunk_text(text, size=800, overlap=150):
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += size - overlap
    return chunks

chunks, meta = [], []
for d in docs:
    for j, ch in enumerate(chunk_text(d['text'])):
        chunks.append(ch)
        meta.append({"doc_id": d['doc_id'], "title": d['title'], "chunk": j})

print(f"{len(docs)} documents → {len(chunks)} chunks")

チャンク数だけ見ても抽象的なので、実物を1つ見てみましょう。**チャンク**とは単なるテキストの窓であり、隣り合うチャンクは**オーバーラップ**しています。境界で切断された文も、どちらかのチャンクには完全な形で現れるためです:

In [ ]:
# find the first pair of consecutive chunks from the SAME document
# (chunks of different documents never overlap — that's case 1 of "why is my overlap empty?")
for i in range(len(chunks) - 1):
    if meta[i]["doc_id"] == meta[i+1]["doc_id"]:
        break

print(f"═══ CHUNK {i} of {meta[i]['doc_id']} ═══\n")
print(chunks[i])
print("\n═══ THE OVERLAP — the same 150 chars live in BOTH chunks ═══")
print("END of chunk", i, ":  …", repr(chunks[i][-150:]))
print("START of chunk", i+1, ":", repr(chunks[i+1][:150]), "…")
print("identical:", chunks[i][-150:] == chunks[i+1][:150])
print(f"\nEach chunk ≈ {len(chunks[i])} chars ≈ {len(enc.encode(chunks[i]))} tokens — small enough to point at one idea, big enough to carry its context. Chunk size is a real tuning knob (Session 2).")

In [ ]:
# --- Embedding (local, free, no API key) ----------------------------------
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions, ~90MB
E = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix:", E.shape)  # (n_chunks, 384)

In [ ]:
# --- Retrieval: cosine similarity = dot product on normalized vectors -----
def retrieve(query, k=4):
    # 1. Embed the query into the SAME 384-dim space as the chunks
    q = embedder.encode([query], normalize_embeddings=True)
    # 2. The actual search is this ONE matrix multiplication:
    #    E is (n_chunks × 384), q.T is (384 × 1) → E @ q.T computes the
    #    dot product of the query against EVERY chunk at once. Normalized
    #    vectors ⇒ dot product == cosine similarity. This is brute-force kNN.
    scores = (E @ q.T).ravel()   # .ravel() merely flattens (n,1)→(n,) — NumPy housekeeping, no magic
    # 3. Rank: indices of the k highest scores
    top = np.argsort(-scores)[:k]
    return [(scores[i], meta[i], chunks[i]) for i in top]

for score, m, ch in retrieve("Why did payments time out in the evening?"):
    print(f"{score:.3f}  {m['doc_id']:12s} {m['title'][:60]}")

👀 検索結果を見てください — ドキュメントとほとんどキーワードを共有しない*言い換え*の質問から、INC-2102 のポストモーテムと payment-gateway のランブックを見つけ出しました。これがセマンティック検索です。では、これを生成につなぎましょう:

In [ ]:
# --- The full RAG loop -----------------------------------------------------
def rag_answer(question, k=4, show_sources=True):
    hits = retrieve(question, k)
    context = "\n\n---\n\n".join(
        f"[Source: {m['doc_id']} — {m['title']}]\n{ch}" for _, m, ch in hits)
    answer = ask_claude(
        f"Answer using ONLY the sources below. Cite source ids. "
        f"If the sources are insufficient, say so.\n\n{context}\n\nQuestion: {question}")
    if show_sources:
        print("Retrieved:", ", ".join(m['doc_id'] for _, m, _ in hits), "\n")
    return answer

print(rag_answer("What was the root cause of the payment gateway timeouts, and what actions were taken?"))

✅ **約40行で動く RAG システムの完成です。** 正しいポストモーテムを取得し、それに基づいて回答し、出典を引用しています。非構造化テキストに対する単一事実の質問なら、このパターンは本番に近い形をしています(セッション2で堅牢化のためのエンジニアリングを足します)。

さて、壊しに行きましょう。😈

---
## 6 · 💥 すべてを破壊する質問

質問の中には、*一節を見つける*ことではなく、**複数ドキュメントにまたがる事実をつなぐ**ことを求めるものがあります。次の質問をゆっくり読んでください。答えるには:

1. どのサービスが `payment-gateway` に**依存している**かを知る(この事実は ADR とサービスカタログにあります)
2. **それらのサービスに影響した**インシデントをすべて見つける(20のポストモーテムに散在)
3. 該当ポストモーテムの対応者(レスポンダー)を**すべて集計**して1つのリストにする

答えを含む単一のチャンクは存在しません。答えとは *JOIN そのもの*なのです。

In [ ]:
HARD_QUESTION = ("Which engineers have responded to incidents affecting services "
                 "that depend on payment-gateway? List all of them.")

print(rag_answer(HARD_QUESTION, k=6))

In [ ]:
# --- Ground truth: don't trust me — COMPUTE it -------------------------------
# The NordWind world ships with its structure as data. Three set operations:
import urllib.request
RELS_URL = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/relationships.json"
rels = json.loads(urllib.request.urlopen(RELS_URL).read())
print(f"{len(rels)} typed relationships, e.g. {rels[40]}\n")

# hop 1: which services DEPEND ON payment-gateway?
dependents = {r["from"] for r in rels if r["type"] == "DEPENDS_ON" and r["to"] == "payment-gateway"}
# hop 2: which incidents AFFECTED those services?
incidents  = {r["from"] for r in rels if r["type"] == "AFFECTED"   and r["to"] in dependents}
# hop 3: which engineers RESPONDED TO those incidents?
engineers  = sorted({r["from"] for r in rels if r["type"] == "RESPONDED_TO" and r["to"] in incidents})

print("hop 1 — services depending on payment-gateway:", sorted(dependents))
print("hop 2 — incidents affecting those services:   ", sorted(incidents))
print(f"hop 3 — the complete correct answer, {len(engineers)} engineers:")
for name in engineers:
    print("   •", name)

Claude の回答を正解(ground truth)と比べてください。典型的には、RAG の回答は**部分的**で(2〜3件のポストモーテムを見つけ、その対応者を列挙しただけ)、**何を見落としているかに気づいておらず**、そして最悪なことに — **自信満々**です。

これはプロンプトの問題ではありません。モデルの問題でもありません。*類似度*による検索は、質問に*似た響きの*チャンクを取ってきました。しかしこの質問が必要としていたのは、*構造で結ばれた*チャンクです: 依存関係のエッジ、インシデントのリンク、所属関係。**意味的類似度は、関係の走査(トラバーサル)ではない。**

この一文をメモしてください。このワークショップ全体のテーゼです。

そして、答えの検証のためにあなたが*たった今やったこと*に注目してください: **型付きの関係**を3ホップたどりました — depends-on、affected、responded-to。気づかないうちに、3行の Python でグラフトラバーサルを書いていたのです。この感覚をセッション3まで持っていてください。

ただし1つ落とし穴があります: これがうまくいったのは、私たちの架空世界がきれいな `relationships.json` を同梱しているからです。**現実の企業にそんなファイルはありません。** 関係は散文の中に埋まっています — このセッションでずっと検索してきた ADR やポストモーテムの中に。非構造化テキストから構造を抽出すること、それこそが GraphRAG の仕事です。セッション4で扱います。

### 🙋 「待って — 65ドキュメント全部を1つのプロンプトに詰めればいいのでは?」

会場で出うる最良の反論です。そして 13K トークンなら答えは: **はい、おそらく動きます。** 下のセルで試してください。その上で計算を見てください: 詰め込みは*質問のたびに*コーパス全量ぶんのコストを払い、検索は一握りのチャンクぶんで済みます — しかも本当のターゲットは 13K ではなく、先ほどの 4000万トークンの赤いバーです。そこでは詰め込みは物理的に不可能で、ウィンドウに収まる範囲ですら *lost in the middle* が静かに品質を蝕みます。構造的な教訓はどのスケールでも生き残ります: **類似度検索は JOIN ができない。** おもちゃのスケールでは詰め込みがそれを覆い隠し、本番のスケールがそれを暴くのです。

In [ ]:
# OPTIONAL — the honest objection, tested (uses ~14K input tokens, one API call)
full_context = "\n\n".join(d['text'] for d in docs)
stuffed = ask_claude(
    f"Answer using ONLY these documents.\n\n<documents>\n{full_context}\n</documents>\n\nQuestion: {HARD_QUESTION}",
    max_tokens=400)
print(stuffed)

avg_chunk = corpus_tokens // len(chunks)
print(f"\n--- the arithmetic ---")
print(f"stuffing:  ~{corpus_tokens:,} input tokens per question")
print(f"RAG (k=6): ~{6*avg_chunk:,} input tokens per question  (~{corpus_tokens//(6*avg_chunk)}× cheaper)")
print(f"at 40M-token scale: stuffing = impossible; retrieval = same cost as today")

---
## 🏁 今日学んだこと

- LLM は**トークン**を読む。コンテキストウィンドウは**作業記憶**であって知識ではない
- 詰め込みはスケールしない(コスト、そして *lost in the middle*)。ファインチューニングは事実を注入できない
- **RAG** = 質問された瞬間に「正しいもの」を取ってくる — そしてそれをゼロから構築した
- ナイーブ RAG は*照会型*の質問には強く、*関係型*の質問で破綻する

## 📝 次回までに
練習用ノートブックにはこのパイプラインに関する演習が5問あります — チャンカーを賢くする問題や、ナイーブ RAG を破壊する**別の**質問を探す問題も(NordWind にはいくつも潜んでいます……)。

---

<div style="background: linear-gradient(120deg,#1e1b4b,#312e81); border-radius:16px; padding:30px 36px; color:#e2e8f0;">
<h2 style="margin:0; color:#ffffff;">⏭️ 次回予告…</h2>
<p style="font-size:1.05em; margin-top:14px;">私たちのリトリーバーは、質問を<b>全チャンクと1つずつ</b>比較しました。500チャンクなら問題ありません。あなたの本番コーパスは<b>500万</b>です。</p>
<p style="font-size:1.05em;">次回: エンベディングとは<i>本当は</i>何なのか(意味の空間をこの目で<b>見ます</b>)、ベクトルデータベースはなぜ全件比較せずに数百万のベクトルをミリ秒で検索できるのか、そしてデモと製品を分かつエンジニアリング — ハイブリッド検索、フィルタリング、リランキング。</p>
<h3 style="color:#93c5fd; margin-bottom:0;">セッション2: エンベディングとベクトルデータベース — <i>意味の幾何学</i> 🌌</h3>
</div>